# Paganin phase retrieval: padding choices

Compare the three original padding strategies on I12 dataset 3. This historical regression deliberately stacks the two raw projections and the flat image; it does not normalise the input and is not a recommended end-to-end phase-retrieval workflow.

Transferred from `zenodo-tests/test_prep/test_phase.py` (SHA-256 `170e0a1ee59a113a4bce410ca2b0e3c95061157b545e3ae9c86cfaf5d707c25e`).
Data: [Zenodo record 17252190](https://zenodo.org/records/17252190).
The original numerical references and tolerances are retained. Figures support visual review;
regression checks are evaluated after all cases so failures retain their figures and measurements.

## Execution

Install `.[validation]` and download the archives with `.scripts/download_zenodo.py Zenodo`.
Set `HTTOMOLIBGPU_DATA_DIR` to use an existing archive directory. Run GPU notebooks serially:

```bash
python -m pytest --nbmake --nbmake-timeout=1800 --overwrite docs/source/validation/preprocessing/paganin_padding.ipynb
```

For a conda CUDA installation, set `CUDA_PATH` to the toolkit root containing
`include/cuda_runtime.h` if CuPy cannot locate the headers. Documentation renders
saved outputs without executing GPU computations.

## Environment and archive loading


In [ ]:
import gc
import hashlib
import json
import os
import platform
from importlib.metadata import version
from pathlib import Path
from math import isclose

import cupy as cp
import matplotlib.pyplot as plt
import numpy as np
import pytest
from numpy.testing import assert_allclose

from httomolibgpu.prep.normalize import dark_flat_field_correction, minus_log

repo_root = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "pyproject.toml").is_file() and (p / "httomolibgpu").is_dir()
    ),
    None,
)
configured_dir = os.environ.get("HTTOMOLIBGPU_DATA_DIR")
if configured_dir:
    data_dir = Path(configured_dir).expanduser()
elif repo_root is not None:
    data_dir = repo_root / "Zenodo"
else:
    raise FileNotFoundError("Set HTTOMOLIBGPU_DATA_DIR to the Zenodo archive directory")
props = cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)
print(
    json.dumps(
        {
            "python": platform.python_version(),
            "packages": {
                name: version(name)
                for name in (
                    "httomolibgpu",
                    "tomobar",
                    "astra-toolbox",
                    "numpy",
                    "matplotlib",
                )
            },
            "cupy": cp.__version__,
            "cuda_runtime": cp.cuda.runtime.runtimeGetVersion(),
            "cuda_driver": cp.cuda.runtime.driverGetVersion(),
            "gpu": (
                props["name"].decode()
                if isinstance(props["name"], bytes)
                else props["name"]
            ),
        },
        indent=2,
    )
)
results = []


def force_clean_gpu_memory():
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()


def load_dataset(name):
    path = data_dir / (name + ".npz")
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}; run .scripts/download_zenodo.py first"
        )
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    print(
        json.dumps(
            {
                "dataset": path.name,
                "zenodo_record": "17252190",
                "sha256": digest.hexdigest(),
            },
            indent=2,
        )
    )
    with np.load(path) as archive:
        if name == "i12_dataset3":
            return tuple(
                cp.asarray(archive[key]) for key in ("proj1", "proj2", "flats", "darks")
            )
        return (
            cp.asarray(archive["projdata"]),
            archive["angles"].copy(),
            cp.asarray(archive["flats"]),
            cp.asarray(archive["darks"]),
        )


def plot_comparison(before, after, title, label="Negative log transmission"):
    before, after = cp.asnumpy(before), cp.asnumpy(after)
    residual = before - after
    lo, hi = np.percentile(before[np.isfinite(before)], [1, 99])
    limit = float(np.percentile(np.abs(residual[np.isfinite(residual)]), 99)) or 1
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    for ax, image, name, cmap, vmin, vmax in (
        (axes[0], before, "Before", "gray", lo, hi),
        (axes[1], after, "After", "gray", lo, hi),
        (axes[2], residual, "Removed component", "coolwarm", -limit, limit),
    ):
        im = ax.imshow(image, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax)
        ax.set(title=name, xlabel="Detector column", ylabel="Projection index")
        fig.colorbar(im, ax=ax, label=label)
    fig.suptitle(title)
    plt.show()
    plt.close(fig)


def check_results():
    print(json.dumps(results, indent=2))
    failed = [row for row in results if not row["passed"]]
    assert not failed, "Transferred regressions failed: " + json.dumps(failed, indent=2)
    print(f"All {len(results)} transferred cases passed.")

In [ ]:
from httomolibgpu.prep.phase import paganin_filter

dataset = load_dataset("i12_dataset3")
inputdata = cp.empty((3, 2050, 2560), dtype=cp.float32)
inputdata[0], inputdata[1], inputdata[2] = dataset[0], dataset[1], dataset[2]
del dataset
force_clean_gpu_memory()
print("Input channels: raw projection 1, raw projection 2, flat image", inputdata.shape)

## Padding comparison

Keep all method defaults except `calculate_padding_value_method` and `pad_x_y`.
The minimum and maximum must match the original references at relative tolerance
`1e-3`, using the original `pytest.approx` comparisons. Only the first projection
is shown; extrema are measured over all three images. Colour scales for each output
are independent because raw intensities and filtered values have different ranges.


In [ ]:
cases = [
    ("next_power_of_2", None, -6.2188172, -10.92260456),
    ("next_fast_length", None, -6.867182, -10.92272),
    ("use_pad_x_y", [80, 80], -6.3293, -10.9227),
]
for padding_method, pad_x_y, expected_max, expected_min in cases:
    output = paganin_filter(
        inputdata, calculate_padding_value_method=padding_method, pad_x_y=pad_x_y
    )
    observed_max = float(cp.max(output).get())
    observed_min = float(cp.min(output).get())
    passed = (
        pytest.approx(observed_max, rel=1e-3) == expected_max
        and pytest.approx(observed_min, rel=1e-3) == expected_min
    )
    results.append(
        {
            "test": "test_paganin_filter_i12_dataset3",
            "padding_method": padding_method,
            "pad_x_y": pad_x_y,
            "observed_max": observed_max,
            "expected_max": expected_max,
            "observed_min": observed_min,
            "expected_min": expected_min,
            "passed": bool(passed),
        }
    )
    print(json.dumps(results[-1], indent=2))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    for ax, image, title in (
        (axes[0], inputdata[0].get(), "Raw projection"),
        (axes[1], output[0].get(), padding_method),
    ):
        lo, hi = np.percentile(image, [1, 99])
        im = ax.imshow(image, cmap="gray", vmin=lo, vmax=hi)
        ax.set(title=title, xlabel="Detector column", ylabel="Detector row")
        fig.colorbar(im, ax=ax)
    plt.show()
    plt.close(fig)
    del output
    force_clean_gpu_memory()
del inputdata
force_clean_gpu_memory()
check_results()